# Processing Raw Sequences

Scripts used to trim raw sequences, check quality, map to ref.

Most of these are bash scripts - written in command line and intended to be run remotely.

Scripts are run out of the `/work` directory and all outputs are routed to my `/scratch` workspace

**steps 1 and 2 can be skipped** - I only need to realign so the steps leading up to that are the same

## 1. trimming 

using `trim-galore` - [documentation](https://github.com/FelixKrueger/TrimGalore)

This code uses an array to run jobs in parallel

In [ ]:
#!/bin/bash
#SBATCH --job-name=trim_galore_array
#SBATCH -c 4
#SBATCH --mem=16G
#SBATCH -p cpu
#SBATCH -t 12:00:00
#SBATCH --array=1-120
#SBATCH -o slurm-%A_%a.out
#SBATCH --mail-type=END,FAIL

#-----------------modules-----------------#
module load conda/latest

conda activate cutadapt
# trim-galore is already installed in this env 

#---------------change wd----------------#

# to scratch workspace with downloaded seqs

cd /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld

#-----------------commands----------------#

# parent directory containing sample subdirectories
PARENT_DIR="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/01.RawData"

# output dir for all trimmed files
OUTDIR="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/trimmed_all"

SAMPLE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/sample_dirs.txt)

R1=("$SAMPLE"/*_1*.fq.gz)
R2=("$SAMPLE"/*_2*.fq.gz)

echo "Running Trim Galore on: $SAMPLE"

trim_galore --paired --fastqc -j 4 -o "$OUTDIR" "${R1[@]}" "${R2[@]}"


## 2. check quality
using FastQC and MultiQC to check quality after trimming adapters

**2a. FastQC** to generate quality assessment files

In [ ]:
#!/bin/bash
#SBATCH --job-name=fastqc_array
#SBATCH -c 4                 # cores per task
#SBATCH --mem=16G             # memory per node
#SBATCH -p cpu
#SBATCH -t 12:00:00
#SBATCH --array=1-120         # number of array tasks
#SBATCH -o slurm-%A_%a.out
#SBATCH --mail-type=END,FAIL

# Load conda and activate environment
module load conda/latest
conda activate fastqc

# Set working directories
INPUT_DIR="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/trimmed_all"
OUTPUT_DIR="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/fastqc"
cd "$INPUT_DIR"

# Number of files per task
FILES_PER_TASK=4

# Compute which lines (files) this array task will process
START=$(( (SLURM_ARRAY_TASK_ID - 1) * FILES_PER_TASK + 1 ))
END=$(( SLURM_ARRAY_TASK_ID * FILES_PER_TASK ))

# Loop over assigned files
for i in $(seq $START $END); do
    FILE=$(sed -n "${i}p" fq_files.txt)
    if [ -n "$FILE" ]; then  # skip if line is empty
        echo "Processing $FILE"
        fastqc -t 2 -o "$OUTPUT_DIR" "$FILE"
    fi
done

**2b. MultiQC** to view all 120 samples (480 files) at once

>multiqc runs quickly, so can be done in terminal and don't need to submit a job. both the html and zip files need to be in the same directory.

In [ ]:
multiqc .

## 3. alignment

using `hisat2` ([manual](https://daehwankimlab.github.io/hisat2/manual/)) - following pipeline from [how to page](https://daehwankimlab.github.io/hisat2/howto/)

(using hisat2 over bowtie2 bc bowtie2 isn't splice aware)

#### 3a. build genome index with exons and splice sites
download reference genome (genome.fa) and GTF file (to make exon, splice site file; genome.gtf)

In [9]:
%cd /work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome
!pwd

/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome
/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome


download fasta file from [haplotig masked genome GitHub repo](https://github.com/The-Eastern-Oyster-Genome-Project/2023_Eastern_Oyster_Haplotig_Masked_Genome) from Jon Puritz

In [10]:
!wget https://github.com/The-Eastern-Oyster-Genome-Project/2022_Eastern_Oyster_Haplotig_Masked_Genome/raw/main/Haplotig_Masking/Output/Masked_Genome/reference.masked.fasta.gz

--2026-05-04 13:42:22--  https://github.com/The-Eastern-Oyster-Genome-Project/2022_Eastern_Oyster_Haplotig_Masked_Genome/raw/main/Haplotig_Masking/Output/Masked_Genome/reference.masked.fasta.gz
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/The-Eastern-Oyster-Genome-Project/2023_Eastern_Oyster_Haplotig_Masked_Genome/raw/main/Haplotig_Masking/Output/Masked_Genome/reference.masked.fasta.gz [following]
--2026-05-04 13:42:22--  https://github.com/The-Eastern-Oyster-Genome-Project/2023_Eastern_Oyster_Haplotig_Masked_Genome/raw/main/Haplotig_Masking/Output/Masked_Genome/reference.masked.fasta.gz
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://media.githubusercontent.com/media/The-Eastern-Oyster-Genome-Project/2023_Eastern_Oyster_Haplotig_Masked_Genome/main/Haplotig

unzip the genome

In [12]:
!gunzip reference.masked.fasta.gz

In [21]:
!ls

2025_CVref
CV_haplotig_masked_ref.fasta
C_virginica.gtf
GCF_002022765.2_C_virginica-3.0_genomic.fna
GCF_002022765.2_C_virginica-3.0_genomic.gff
GCF_002022765.2_C_virginica-3.0_genomic.gtf
cv.gff
fastq-screen-genomes
reference.masked.fasta


make exon and splice sites files from GTF file
>yea i guess i'm confused on how to do this part now since i need an annotation file - i'm waiting to hear back from Jon about which annotation file he uses, but I guess I can assume for right now it's the same annotation file as before?

running in command line since I need the `rnaseq-env` environment

In [ ]:
# Extract splice sites
hisat2_extract_splice_sites.py GCF_002022765.2_C_virginica-3.0_genomic.gff > oyster.ss

# Extract exons
hisat2_extract_exons.py GCF_002022765.2_C_virginica-3.0_genomic.gff > oyster.exon


Build HFM index - with exon and splice site info using the files above

(HFM = hierarchical FM index for a reference genome)

In [ ]:
#!/bin/bash
#SBATCH --job-name=hisat2_build
#SBATCH --mem=32G             # memory per node
#SBATCH -p cpu
#SBATCH -t 1:00:00
#SBATCH --cpus-per-task=16     
#SBATCH -o hisat2_build.log
#SBATCH --mail-type=END,FAIL

#-----------------modules-----------------#
module load conda/latest

conda activate rnaseq-env

#---------------change wd----------------#

cd /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/haplotig-masked

#-----------------commands----------------#

hisat2-build -p 16 \
--exon /work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/oyster.exon \
--ss /work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/oyster.ss \
/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/reference.masked.fasta oyster_index

then need to move all indexes to other directory

In [ ]:
mv *.ht2 /work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/hisatIndex-haplotigMasked/

#### 3b. align sequences to genome index

the code I used in the last alignment took about 32 hours to finish - below is code for running multiple alignments in parallel with an array job (took about 23 hours)

also directly converts to BAM files so we can skip the SAM step, which creates HUGE files - so this should be much faster!

In [ ]:
#!/bin/bash
#SBATCH --job-name=hisat2_align
#SBATCH --cpus-per-task=16
#SBATCH --mem=32G
#SBATCH -p cpu
#SBATCH -t 32:00:00
#SBATCH -o hisat2_align_%j.log
#SBATCH --mail-type=END,FAIL

set -euo pipefail

module load conda/latest
conda activate rnaseq-env # need hisat2 and samtools

INDEX="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/haplotig-masked/oyster_index"
INPUT_DIR="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/trimmed_all"
OUTPUT_DIR="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/haplotig-masked"

mkdir -p "$OUTPUT_DIR"

FILES=(${INPUT_DIR}/*_gi_1_val_1.fq.gz)

for R1 in "${FILES[@]}"; do
    SAMPLE=$(basename "$R1" _gi_1_val_1.fq.gz)
    R2="${INPUT_DIR}/${SAMPLE}_gi_2_val_2.fq.gz"

    [[ -f "$R2" ]] || { echo "Missing $R2"; continue; }

    echo "Aligning $SAMPLE"

    hisat2 -p 12 \
        -x $INDEX \
        -1 "$R1" \
        -2 "$R2" \
        2> "${OUTPUT_DIR}/${SAMPLE}.log" | \
    samtools sort -@ 4 -o "${OUTPUT_DIR}/${SAMPLE}.bam"

done

from a very precursory look, alignment rates are reduced about 10-12% from the old reference - this might be okay, but will need to check that they're universally down for all samples (and therefore likely reference related) or if some samples are more reduced than others (likely contamination/QC issue then). 

## 4. read counting
with `Subread-featureCounts' ([manual](https://subread.sourceforge.net/featureCounts.html)) to generate counts matrix


featureCounts recommends to use the same GTF/GFF file that was used for alignment - so using GFF file, but removing the first couple of lines that start with "#" so that featureCounts can read it correctly


**this has already been done, so i'm going to use the original file instead of overwriting it**

In [ ]:
# navigate to directory
#cd /work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome

# remove #s in first couple of lines of gff file
# grep -v '^#' GCF_053477285.1_ASM5347728v1_genomic.gff > cv.gff

to run the code in 4a, we'll want to provide a list of all the BAM files we need to process:

In [ ]:
ls *.bam > bam_files.txt

### 4a. running featureCounts
keeping the same settings that were used with the old reference genome

In [ ]:
#!/bin/bash
#SBATCH --job-name=featureCounts
#SBATCH --array=1-120          
#SBATCH --cpus-per-task=8      # more threads for featureCounts
#SBATCH --mem=16G               
#SBATCH -p cpu
#SBATCH -t 1:00:00              
#SBATCH --output=logs/featureCounts_%A_%a.log
#SBATCH --mail-type=END,FAIL
#-----------------modules-----------------#

module load conda/latest
conda activate subread_env

#-----------------set paths----------------#

INPUT_DIR="/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/haplotig-masked"
OUTPUT_DIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ce24_rnaseq/haplotig_masked/featureCounts"

BAM_LIST="$INPUT_DIR/bam_files.txt"

GFF_FILE="/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/cv.gff"

#-----------------commands----------------#

# Get the BAM file for this array task
BAM_FILE="$INPUT_DIR/$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$BAM_LIST")"

SAMPLE_NAME=$(basename "$BAM_FILE" .sorted.bam)

# Run featureCounts
featureCounts \
-T 8 \
-a "$GFF_FILE" \
-g ID \
-t gene \
-p \
-B \
-C \
-s 0 \
-o "$OUTPUT_DIR/counts_${SAMPLE_NAME}.txt" \
"$BAM_FILE"

annotation of featureCounts options:
```
featureCounts \
  -T 8 \                  # threads
  -a "$gff.file" \        # GTF annotation
  -g ID \                 # gene identifier attribute
  -t gene \               # feature type
  -p \                    # paired-end
  -B \                    # require both ends mapped
  -C \                    # ignore chimeric fragments
  -s 0 \                  # unstranded
  -o "$OUTPUT_DIR/counts_${SLURM_ARRAY_TASK_ID}.txt" \
  "$BAM_FILE"
```

the output of this will be multiple .txt files, two for each sample (one containing the counts matrix, the other containing summaries about read counting)

### 4b. combining output files
first for .summary files, which has info regarding number of reads assigned for read counting

In [ ]:
INPUT_DIR="/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ce24_rnaseq/newRef_featureCounts/"
OUTPUT_FILE="/project/pi_sarah_gignouxwolfsohn_uml_edu/julia/CE_2024/CE24_RNA-seq/processing/qc_outputs/featureCounts_summary_newRef.csv"

# Get header from the first file
first_file=$(ls $INPUT_DIR/*.summary | head -n 1)
awk 'NR>1 {print $1}' "$first_file" | paste -sd ',' - > "$OUTPUT_FILE.header"

# Start the CSV with "Sample" column
echo "Sample,$(cat $OUTPUT_FILE.header)" > "$OUTPUT_FILE"

# Loop over all files
for f in $INPUT_DIR/*.summary; do
    sample=$(basename "$f" .txt.summary)
    sample=${sample%.bam}
    awk 'NR>1 {print $2}' "$f" | paste -sd ',' - | awk -v s="$sample" '{print s "," $0}' >> "$OUTPUT_FILE"
done

# Clean up
rm "$OUTPUT_FILE.header"

then for the counts matrix (easier to do in R)

In [2]:
library(dplyr)
library(readr)
library(stringr)
library(tidyr)

In [3]:
files <- list.files("//work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ce24_rnaseq/newRef_featureCounts", 
                    pattern="counts_.*\\.txt$", 
                    full.names = TRUE)

all_counts <- lapply(files, function(f) {
  sample_name <- gsub(".txt$", "", basename(f))
  df <- read_tsv(f, comment="#", show_col_types = FALSE)
  
  # select Geneid, Length, and the count column
  df %>% 
    select(Geneid, Length, Count = 7) %>% 
    rename(!!sample_name := Count)
})

# Reduce using full_join by Geneid and Length
counts_matrix <- Reduce(function(x, y) full_join(x, y, by = c("Geneid", "Length")), all_counts)

head(counts_matrix)

Geneid,Length,counts_B1_B1_O01.bam,counts_B1_Nu_O03.bam,counts_B1_W5_O50.bam,counts_B2_B5_O51.bam,counts_B2_C4_O40.bam,counts_B2_Nu_O12.bam,counts_B3_B4_O41.bam,counts_B3_C3_O30.bam,⋯,counts_W5_C4_G45.bam,counts_W5_H4_G46.bam,counts_W5_W2_G22.bam,counts_W6_B3_G35.bam,counts_W6_B4_G48.bam,counts_W6_H6_G71.bam,counts_W6_Nu_G41.bam,counts_W6_Nu_G45.bam,counts_W6_W3_G36.bam,counts_W6_W4_G48.bam
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
gene-LOC144621260,36658,2626,2649,2072,2158,2074,3974,1207,1181,⋯,1252,2294,2215,1653,1720,1566,2734,2374,2683,2391
gene-LOC144621269,25160,55,580,1632,2484,1011,20011,2110,886,⋯,6,620,44,101,9226,16,2,4034,383,9803
gene-LOC111120925,3392,70,4,23,469,11,18,15,7,⋯,3,452,82,28,5,1,0,11,11,9
gene-Trnae-cuc,72,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
gene-Trnae-cuc-2,72,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
gene-Trnae-cuc-3,72,0,0,0,0,0,1,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [4]:
## df formatting

# remove 'counts_' prefix from column name
colnames(counts_matrix)[-(1:2)] <- str_remove(colnames(counts_matrix)[-(1:2)], "^counts_")

# change first column name
colnames(counts_matrix)[1] <- 'Gene_ID'

# remove gene- in geneid column
counts_matrix$Gene_ID <- gsub('gene-', '', counts_matrix$Gene_ID)

# remove .bam from col names
colnames(counts_matrix) <- sub("\\.bam$", "", colnames(counts_matrix))

head(counts_matrix)

Gene_ID,Length,B1_B1_O01,B1_Nu_O03,B1_W5_O50,B2_B5_O51,B2_C4_O40,B2_Nu_O12,B3_B4_O41,B3_C3_O30,⋯,W5_C4_G45,W5_H4_G46,W5_W2_G22,W6_B3_G35,W6_B4_G48,W6_H6_G71,W6_Nu_G41,W6_Nu_G45,W6_W3_G36,W6_W4_G48
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
LOC144621260,36658,2626,2649,2072,2158,2074,3974,1207,1181,⋯,1252,2294,2215,1653,1720,1566,2734,2374,2683,2391
LOC144621269,25160,55,580,1632,2484,1011,20011,2110,886,⋯,6,620,44,101,9226,16,2,4034,383,9803
LOC111120925,3392,70,4,23,469,11,18,15,7,⋯,3,452,82,28,5,1,0,11,11,9
Trnae-cuc,72,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
Trnae-cuc-2,72,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
Trnae-cuc-3,72,0,0,0,0,0,1,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [6]:
write_csv(counts_matrix, "/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ce24_rnaseq/newRef_featureCounts/featureCounts_matrix.csv")

## 5. fastQ screen
[fastQ screen documentation](https://www.bioinformatics.babraham.ac.uk/projects/fastq_screen/) - this will allow me to see if my unmapped reads align to other genomes (like human or bacteria) to look for possible contamination
- [bioconda installer](https://bioconda.github.io/recipes/fastq-screen/README.html)

In [ ]:
# create new environment, install fastq-screen
conda create -n fastq-screen fastq-screen

### 5a. get reference genomes
the program suggests human, E. coli, rRNA, and PhiX (and of course, oyster)

some of these are already on the cluster, some I downloaded from NCBI

**from Unity:**
- human: `/datasets/bio/gmap-gsnap/2025-01-02/GCF_000001405.40_GRCh38.p14_genomic.fna`

**from NCBI:**
- *Crassostrea virginica*: `/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/cvirginica.fna`
  - [mitochondrial genome](https://www.ncbi.nlm.nih.gov/nuccore/AY905542.2/): `/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/CV_mitochondrial.fna`
- E. coli: `/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/ecoli.fna`
- rRNA: `/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/rrna.fna`
- PhiX: `/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/phix.fna`
- *Perkinsus marinus*: `/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/perkinsus.fna`
- *Vibrio vulnificus*: `/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/vibrio.fna`

In [ ]:
# C. virginica
# downloaded from NCBI
conda activate ncbi-datasets-cli

datasets download genome accession GCF_053477285.1 --include gff3,rna,cds,protein,genome,seq-report

In [ ]:
# E. coli
# downloaded from NCBI
datasets download genome accession GCF_000019425.1 \
  --include genome

In [ ]:
# PhiX
# downloaded from NCBI
wget https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/819/615/GCF_000819615.1_ViralProj14015/GCF_000819615.1_ViralProj14015_genomic.fna.gz
gunzip *.gz
mv GCF_000819615.1_ViralProj14015_genomic.fna phix.fa

In [ ]:
# rRNA
# downloaded from NCBI
wget https://ftp.arb-silva.de/release_138/Exports/SILVA_138_SSURef_NR99_tax_silva.fasta.gz
gunzip SILVA_138_SSURef_NR99_tax_silva.fasta.gz
mv SILVA_138_SSURef_NR99_tax_silva.fasta rrna.fa

In [ ]:
# perkinsus marinus
# downloaded from NCBI
datasets download genome accession GCF_000006405.1 --include genome

In [ ]:
# vibrio vulnificus
# downloaded from NCBI
datasets download genome accession GCF_002224265.1 --include genome

### 5b. build `bowtie` indices
unfortunately, fastq-screen is not compatible with `hisat2`, so I have to use `bowtie2` ... just something to be aware of

In [ ]:
#!/bin/bash
#SBATCH --job-name=bowtie2_index
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH --array=0-4
#SBATCH --time=12:00:00
#SBATCH --output=index_%A_%a.log

##---------load modules---------##
module load conda/latest
conda activate fastq-screen

##---------inputs---------##

THREADS=8

# INPUT FASTA FILES (EDIT THESE)
FASTA_FILES=(
/datasets/bio/gmap-gsnap/2025-01-02/GCF_000001405.40_GRCh38.p14_genomic.fna
/datasets/bio/ncbi-refseq/bacterial_genomes/1_genomes/ncbi_dataset/data/GCF_000019425.1/GCF_000019425.1_ASM1942v1_genomic.fna
/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/cvirginica.fna
/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/phix.fa
/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/rrna.fa
/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/CV_mitochondrial.fna
/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/perkinsus.fna
/work/pi_sarah_gignouxwolfsohn_uml_edu/julia_mcdonough_student_uml_edu/ref_files/genome/fastq-screen-genomes/vibrio.fna
)

# OUTPUT PREFIX NAMES (EDIT IF YOU WANT DIFFERENT NAMES)
PREFIXES=(
human
ecoli
oyster
phix
rrna
oyst_mito
perkinsus
vibrio
)

# OUTPUT DIRECTORY FOR INDEX FILES
OUTDIR=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices

##---------commands---------##

INPUT=${FASTA_FILES[$SLURM_ARRAY_TASK_ID]}
PREFIX=${PREFIXES[$SLURM_ARRAY_TASK_ID]}

echo "======================================"
echo "Building index for: $PREFIX"
echo "Input FASTA: $INPUT"
echo "Output dir: $OUTDIR"
echo "Threads: $THREADS"
echo "======================================"

bowtie2-build --threads $THREADS \
  "$INPUT" \
  "$OUTDIR/$PREFIX"

echo "DONE: $PREFIX"

In [ ]:
scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/vibrio.4.bt2

### 5c. create `fastq_screen.conf`

In [ ]:
ALIGNER bowtie2

DATABASE Human   /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/human
DATABASE Ecoli   /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/ecoli
DATABASE Oyster  /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/oyster
DATABASE PhiX    /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/phix
DATABASE rRNA    /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/rrna
DATABASE OysterMitochondrial    /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/oyst_mito
DATABASE Perkinsus    /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/perkinsus
DATABASE Vibrio    /scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/vibrio

### 5d. run `fastq_screen`

In [ ]:
#!/bin/bash
#SBATCH --job-name=fastq_screen
#SBATCH --cpus-per-task=8
#SBATCH --mem=16G
#SBATCH --time=04:00:00
#SBATCH --array=1-120
#SBATCH --output=logs/fastq_screen_%A_%a.log


##---------load modules---------##
module load conda/latest
conda activate fastq-screen

##---------inputs---------##

THREADS=8

# File listing samples (name R1 R2)
SAMPLE_FILE=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/trimmed_all/samples.txt

# Output directory for FastQ Screen results
OUTDIR=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/fastq_screen_res

# FastQ Screen config (your indexes live here)
CONFIG=/scratch4/workspace/julia_mcdonough_student_uml_edu-novogene_dwnld/bowtie-indices/fastq_screen.conf


##---------commands---------##

# Get the line for this array job
LINE=$(sed -n "${SLURM_ARRAY_TASK_ID}p" "$SAMPLE_FILE")

SAMPLE=$(echo "$LINE" | awk '{print $1}')
R1=$(echo "$LINE" | awk '{print $2}')
R2=$(echo "$LINE" | awk '{print $3}')

echo "======================================"
echo "Sample: $SAMPLE"
echo "R1: $R1"
echo "R2: $R2"
echo "Output: $OUTDIR"
echo "======================================"

# run fastq screen
fastq_screen \
  --conf "$CONFIG" \
  --outdir "$OUTDIR" \
  --threads "$THREADS" \
  "$R1" "$R2"

echo "DONE: $SAMPLE"

apparently `fastq_screen` is compatible with [`multiqc`](https://seqera.io/multiqc/) which is fantastic news

In [ ]:
# run in the same directory as the fastq-screen results
multiqc .